<center><h1><span style="color: green;">THỰC HÀNH BUỔI 8.2</span></h1></center>

# Bài tập 1

### 1. **Huấn luyện và dự đoán đầu ra của dữ liệu trên cả hai tập, đánh giá độ chính xác của mô hình thông qua các độ đo $R2 (R – squared)$ và $MSE$.**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [5]:
# Đọc dữ liệu
data = pd.read_csv("SAT_GPA.csv")
X = data[["SAT"]].values
y = data[["GPA"]].values

# Chuẩn hóa dữ liệu
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Chia dữ liệu
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled, train_size=54, random_state=42)

# Chuyển dữ liệu thành dạng ma trận
X_train = X_train.T  # Shape: (1, 54)
y_train = y_train.T  # Shape: (1, 54)
X_val = X_val.T      # Shape: (1, 30)
y_val = y_val.T      # Shape: (1, 30)

In [2]:
# Hàm ReLU và đạo hàm
def relu(Z):
    return np.maximum(0, Z)

def relu_deriv(Z):
    return Z > 0

# Hàm kích hoạt đầu ra (tuyến tính)
def linear(Z):
    return Z

# Hàm loss MSE
def mse_loss(Y, Yhat):
    return np.mean((Y - Yhat) ** 2)

# Đạo hàm của MSE
def mse_loss_deriv(Y, Yhat):
    return Yhat - Y  # f'(z) = 1, nên gradient là (ŷ - y)

In [4]:
# Khởi tạo tham số
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    W1 = np.random.randn(hidden_size, input_size) * 0.01
    b1 = np.zeros((hidden_size, 1))
    W2 = np.random.randn(output_size, hidden_size) * 0.01
    b2 = np.zeros((output_size, 1))
    return W1, b1, W2, b2

# Forward propagation
def forward_propagation(X, W1, b1, W2, b2):
    Z1 = W1 @ X + b1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + b2
    A2 = linear(Z2)  # Đầu ra tuyến tính
    return Z1, A1, Z2, A2

# Backward propagation
def backward_propagation(X, Y, Z1, A1, Z2, A2, W1, W2):
    m = X.shape[1]
    dZ2 = mse_loss_deriv(Y, A2)  # Đạo hàm MSE
    dW2 = (1/m) * (dZ2 @ A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dA1 = W2.T @ dZ2
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = (1/m) * (dZ1 @ X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

# Cập nhật tham số
def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    return W1, b1, W2, b2

In [7]:
# Huấn luyện mô hình
def train_model(X, Y, hidden_size, num_iterations, learning_rate):
    input_size = X.shape[0]
    output_size = Y.shape[0]
    W1, b1, W2, b2 = initialize_parameters(input_size, hidden_size, output_size)
    
    for i in range(num_iterations):
        Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)
        loss = mse_loss(Y, A2)
        dW1, db1, dW2, db2 = backward_propagation(X, Y, Z1, A1, Z2, A2, W1, W2)
        W1, b1, W2, b2 = update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)
        
        if i % 1000 == 0:
            print(f"Iteration {i}, Loss: {loss}")
    
    return W1, b1, W2, b2

# Dự đoán
def predict(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_propagation(X, W1, b1, W2, b2)
    return A2

In [8]:
# Huấn luyện mô hình
hidden_size = 10
num_iterations = 5000
learning_rate = 0.01
W1, b1, W2, b2 = train_model(X_train, y_train, hidden_size, num_iterations, learning_rate)

# Dự đoán trên tập huấn luyện và validation
y_train_pred_scaled = predict(X_train, W1, b1, W2, b2)
y_val_pred_scaled = predict(X_val, W1, b1, W2, b2)

# Chuyển ngược dữ liệu về thang ban đầu
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.T).T
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.T).T
y_train_true = scaler_y.inverse_transform(y_train.T).T
y_val_true = scaler_y.inverse_transform(y_val.T).T

# Đánh giá mô hình
train_mse = mean_squared_error(y_train_true.T, y_train_pred.T)
val_mse = mean_squared_error(y_val_true.T, y_val_pred.T)
train_r2 = r2_score(y_train_true.T, y_train_pred.T)
val_r2 = r2_score(y_val_true.T, y_val_pred.T)

print("\nKết quả đánh giá:")
print(f"Tập huấn luyện - MSE: {train_mse:.4f}, R²: {train_r2:.4f}")
print(f"Tập validation - MSE: {val_mse:.4f}, R²: {val_r2:.4f}")

Iteration 0, Loss: 0.9861818279013259
Iteration 1000, Loss: 0.7462157824761696
Iteration 2000, Loss: 0.5771174705382282
Iteration 3000, Loss: 0.5761572232833406
Iteration 4000, Loss: 0.5754659635207152

Kết quả đánh giá:
Tập huấn luyện - MSE: 0.0419, R²: 0.4134
Tập validation - MSE: 0.0451, R²: 0.3851


### Nhận xét
- **Loss**: Giảm ổn định từ 0.986 xuống 0.575, mô hình gần hội tụ sau 4000 lần lặp.
- **Tập huấn luyện**: MSE 0.0419, R² 0.4134 – mô hình giải thích ~41% biến thiên GPA, hiệu suất trung bình.
- **Tập validation**: MSE 0.0451, R² 0.3851 – gần với tập huấn luyện, không bị overfitting.
- **Tổng thể**: Mô hình ổn, nhưng R² thấp cho thấy SAT không đủ mạnh để dự đoán GPA chính xác.

### 2. **Sử dụng mô hình hồi quy tuyến tính để thực hiện lại dự đoán. So sánh với mô hình ANN trên các tiêu chí: Thời gian training; Thời gian predict (tính trung bình); độ chính xác.**

In [13]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [12]:
# Huấn luyện mô hình hồi quy tuyến tính
start_train_lr = time.time()
model_lr = LinearRegression()
model_lr.fit(X_train.T, y_train.T)  # Chuyển về dạng (n_samples, n_features)
end_train_lr = time.time()
train_time_lr = end_train_lr - start_train_lr

# Dự đoán và đo thời gian trung bình
start_pred_train_lr = time.time()
y_train_pred_scaled_lr = model_lr.predict(X_train.T)
end_pred_train_lr = time.time()
pred_time_train_lr = (end_pred_train_lr - start_pred_train_lr) / X_train.shape[1]

start_pred_val_lr = time.time()
y_val_pred_scaled_lr = model_lr.predict(X_val.T)
end_pred_val_lr = time.time()
pred_time_val_lr = (end_pred_val_lr - start_pred_val_lr) / X_val.shape[1]
avg_pred_time_lr = (pred_time_train_lr + pred_time_val_lr) / 2

# Chuyển ngược về thang gốc
y_train_pred_lr = scaler_y.inverse_transform(y_train_pred_scaled_lr)
y_val_pred_lr = scaler_y.inverse_transform(y_val_pred_scaled_lr)
y_train_true_lr = scaler_y.inverse_transform(y_train.T)
y_val_true_lr = scaler_y.inverse_transform(y_val.T)

# Đánh giá mô hình hồi quy tuyến tính
train_mse_lr = mean_squared_error(y_train_true_lr, y_train_pred_lr)
val_mse_lr = mean_squared_error(y_val_true_lr, y_val_pred_lr)
train_r2_lr = r2_score(y_train_true_lr, y_train_pred_lr)
val_r2_lr = r2_score(y_val_true_lr, y_val_pred_lr)

# In kết quả hồi quy tuyến tính
print("\nHồi quy tuyến tính:")
print(f"Thời gian huấn luyện: {train_time_lr:.6f} giây")
print(f"Thời gian dự đoán trung bình: {avg_pred_time_lr:.6f} giây/mẫu")
print(f"Tập huấn luyện - MSE: {train_mse_lr:.4f}, R²: {train_r2_lr:.4f}")
print(f"Tập validation - MSE: {val_mse_lr:.4f}, R²: {val_r2_lr:.4f}")


Hồi quy tuyến tính:
Thời gian huấn luyện: 0.007403 giây
Thời gian dự đoán trung bình: 0.000025 giây/mẫu
Tập huấn luyện - MSE: 0.0434, R²: 0.3927
Tập validation - MSE: 0.0455, R²: 0.3801


#### Nhận xét:
- **Hồi quy tuyến tính**:
  - **Thời gian**: Huấn luyện rất nhanh (0.0074 giây), dự đoán cực kỳ hiệu quả (0.000025 giây/mẫu).
  - **Độ chính xác**: MSE (train: 0.0434, val: 0.0455) và R² (train: 0.3927, val: 0.3801) cho thấy hiệu suất trung bình, tương đương ANN nhưng thấp hơn một chút.
- **So sánh với ANN** (MSE train: 0.0419, R² train: 0.4134; MSE val: 0.0451, R² val: 0.3851):
  - Hồi quy tuyến tính nhanh hơn đáng kể về thời gian huấn luyện và dự đoán.
  - ANN nhỉnh hơn về độ chính xác (R² cao hơn ~0.02), nhưng không đáng kể so với tốc độ của hồi quy tuyến tính.
- **Kết luận**: Hồi quy tuyến tính phù hợp hơn cho bài toán này do tốc độ vượt trội và độ chính xác gần tương đương.

### 3. **Thay đổi số chiều layer ẩn lần lượt là 75, 50. Thực nghiệm lại và đánh giá sự thay đổi kết quả so với các thông số trong đoạn code đã cho. Hãy cho nhận xét về mối liên hệ giữa siêu tham số trên với kết quả dự đoán.**

In [14]:
# Thực nghiệm với các hidden_size khác nhau
hidden_sizes = [75, 50]
results = {}

for hidden_size in hidden_sizes:
    print(f"\nThực nghiệm với hidden_size = {hidden_size}")
    # Huấn luyện mô hình
    W1, b1, W2, b2 = train_model(X_train, y_train, hidden_size, num_iterations=5000, learning_rate=0.01)
    
    # Dự đoán
    y_train_pred_scaled = predict(X_train, W1, b1, W2, b2)
    y_val_pred_scaled = predict(X_val, W1, b1, W2, b2)
    
    # Chuyển ngược về thang gốc
    y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.T).T
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.T).T
    y_train_true = scaler_y.inverse_transform(y_train.T).T
    y_val_true = scaler_y.inverse_transform(y_val.T).T
    
    # Đánh giá
    train_mse = mean_squared_error(y_train_true.T, y_train_pred.T)
    val_mse = mean_squared_error(y_val_true.T, y_val_pred.T)
    train_r2 = r2_score(y_train_true.T, y_train_pred.T)
    val_r2 = r2_score(y_val_true.T, y_val_pred.T)
    
    results[hidden_size] = {
        "train_mse": train_mse,
        "val_mse": val_mse,
        "train_r2": train_r2,
        "val_r2": val_r2
    }
    
    print(f"Tập huấn luyện - MSE: {train_mse:.4f}, R²: {train_r2:.4f}")
    print(f"Tập validation - MSE: {val_mse:.4f}, R²: {val_r2:.4f}")

# In kết quả so sánh với hidden_size = 10 (ban đầu)
print("\nSo sánh với hidden_size = 10 (ban đầu):")
print(f"Tập huấn luyện - MSE: 0.0419, R²: 0.4134")
print(f"Tập validation - MSE: 0.0451, R²: 0.3851")


Thực nghiệm với hidden_size = 75
Iteration 0, Loss: 0.9862054622270208
Iteration 1000, Loss: 0.583933536927697
Iteration 2000, Loss: 0.5750694647993834
Iteration 3000, Loss: 0.5715998684015644
Iteration 4000, Loss: 0.5672566531271409
Tập huấn luyện - MSE: 0.0413, R²: 0.4218
Tập validation - MSE: 0.0448, R²: 0.3899

Thực nghiệm với hidden_size = 50
Iteration 0, Loss: 0.9856438931817557
Iteration 1000, Loss: 0.5893380137905556
Iteration 2000, Loss: 0.5758975677441674
Iteration 3000, Loss: 0.5751185723617569
Iteration 4000, Loss: 0.5736131098550225
Tập huấn luyện - MSE: 0.0416, R²: 0.4182
Tập validation - MSE: 0.0450, R²: 0.3862

So sánh với hidden_size = 10 (ban đầu):
Tập huấn luyện - MSE: 0.0419, R²: 0.4134
Tập validation - MSE: 0.0451, R²: 0.3851


#### Nhận xét:
- **Hiệu suất**:
  - `hidden_size = 75`: MSE train giảm (0.0413 vs. 0.0419), R² train tăng (0.4218 vs. 0.4134); validation cải thiện nhẹ (MSE: 0.0448, R²: 0.3899).
  - `hidden_size = 50`: Kết quả gần tương tự, nhưng kém hơn chút so với 75 (MSE train: 0.0416, R² train: 0.4182; MSE val: 0.0450, R² val: 0.3862).
  - So với `hidden_size = 10`: Cả 75 và 50 đều cải thiện nhẹ trên train và validation, nhưng mức tăng nhỏ.
- **Mối liên hệ**:
  - Tăng `hidden_size` giúp mô hình học tốt hơn trên train, nhưng cải thiện trên validation không đáng kể, cho thấy dữ liệu SAT-GPA có mối quan hệ đơn giản.
  - `hidden_size = 75` tốt nhất, nhưng chi phí tính toán cao hơn không tương xứng với cải thiện.
- **Kết luận**: `hidden_size = 10` đủ hiệu quả cho bài toán này, cân bằng giữa độ chính xác và tài nguyên.

# Bài tập 2

_Về tính toán bề dày lớp nội trung mạc (NTM) – thuộc tính phản ánh một số bệnh lý của cơ thể. Trong thực tế hiện tượng dày lớp NTM động mạch cảnh do nhiều yếu tố như di truyền, chủng tộc, mắc bệnh tim mạch, tuổi, giới, BMI, tăng huyết áp, đái tháo đường.... cùng tác động. Trong ví dụ này ta không đề cập các yếu tố di truyền, chủng tộc, giới, mắc bệnh tim mạch... mà chỉ lưu ý đến các biến số như: tuổi, cholesterol, glucose, huyết áp tâm thu và BMI tác động lên độ dày NTM._

**Áp dụng mô hình ANN và dữ liệu cho trong tệp vidu4_lin_reg.txt để dự đoán bề dày lớp NTM theo các biến số khác.**

### Chia dữ liệu thành: 80 dòng đầu dùng cho training; 20 dòng sau dùng cho testing. Huấn luyện mô hình ANN cải biên với phần training và thực hiện dự đoán trên cả 2 tập.

In [17]:
# Đọc dữ liệu
data = pd.read_csv("vidu4_lin_reg.txt", delim_whitespace=True)

# Tách đặc trưng và nhãn
X = data[["TUOI", "BMI", "HA", "GLUCOSE", "CHOLESTEROL"]].values
y = data[["BEDAYNTM"]].values

# Chuẩn hóa dữ liệu
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Chia dữ liệu: 80 dòng đầu cho train, 20 dòng sau cho test
X_train = X_scaled[:80]
y_train = y_scaled[:80]
X_test = X_scaled[80:]
y_test = y_scaled[80:]

# Chuyển dữ liệu thành dạng ma trận
X_train = X_train.T  # Shape: (5, 80)
y_train = y_train.T  # Shape: (1, 80)
X_test = X_test.T    # Shape: (5, 20)
y_test = y_test.T    # Shape: (1, 20)

In [18]:
# Huấn luyện mô hình
hidden_size = 10
num_iterations = 5000
learning_rate = 0.01
W1, b1, W2, b2 = train_model(X_train, y_train, hidden_size, num_iterations, learning_rate)

# Dự đoán
y_train_pred_scaled = predict(X_train, W1, b1, W2, b2)
y_test_pred_scaled = predict(X_test, W1, b1, W2, b2)

# Chuyển ngược về thang gốc
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.T).T
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.T).T
y_train_true = scaler_y.inverse_transform(y_train.T).T
y_test_true = scaler_y.inverse_transform(y_test.T).T

# Đánh giá
train_mse = mean_squared_error(y_train_true.T, y_train_pred.T)
test_mse = mean_squared_error(y_test_true.T, y_test_pred.T)
train_r2 = r2_score(y_train_true.T, y_train_pred.T)
test_r2 = r2_score(y_test_true.T, y_test_pred.T)

print("\nKết quả đánh giá:")
print(f"Tập huấn luyện - MSE: {train_mse:.4f}, R²: {train_r2:.4f}")
print(f"Tập kiểm tra - MSE: {test_mse:.4f}, R²: {test_r2:.4f}")

Iteration 0, Loss: 0.799433706809961
Iteration 1000, Loss: 0.7829449786437158
Iteration 2000, Loss: 0.6329425455903255
Iteration 3000, Loss: 0.5728352212083783
Iteration 4000, Loss: 0.5349085896179997

Kết quả đánh giá:
Tập huấn luyện - MSE: 0.0836, R²: 0.3517
Tập kiểm tra - MSE: 0.2361, R²: 0.1846


#### Nhận xét:
- **Quá trình huấn luyện**: Loss giảm từ 0.7994 xuống 0.5349, mô hình học tốt trên tập huấn luyện nhưng có dấu hiệu hội tụ chậm.
- **Hiệu suất**:
  - **Tập huấn luyện**: MSE 0.0836, R² 0.3517 – mô hình giải thích ~35% biến thiên BEDAYNTM, hiệu quả trung bình.
  - **Tập kiểm tra**: MSE 0.2361, R² 0.1846 – hiệu suất thấp, chênh lệch lớn so với train, cho thấy mô hình bị overfitting.
- **Kết luận**: Mô hình ANN chưa tổng quát hóa tốt, cần điều chỉnh cấu trúc (tăng nơ-ron/tầng ẩn) hoặc siêu tham số (learning rate, iterations) để cải thiện trên tập kiểm tra.

### So sánh kết quả này với kết quả phương pháp hồi quy tuyến tính

In [22]:
# Huấn luyện mô hình hồi quy tuyến tính
start_train_lr = time.time()
model_lr = LinearRegression()
model_lr.fit(X_train.T, y_train.T)  # Chuyển về dạng (n_samples, n_features)
end_train_lr = time.time()
train_time_lr = end_train_lr - start_train_lr

# Dự đoán
y_train_pred_scaled_lr = model_lr.predict(X_train.T)
y_test_pred_scaled_lr = model_lr.predict(X_test.T)

# Chuyển ngược về thang gốc
y_train_pred_lr = scaler_y.inverse_transform(y_train_pred_scaled_lr)
y_test_pred_lr = scaler_y.inverse_transform(y_test_pred_scaled_lr)
y_train_true_lr = scaler_y.inverse_transform(y_train.T)
y_test_true_lr = scaler_y.inverse_transform(y_test.T)

# Đánh giá mô hình hồi quy tuyến tính
train_mse_lr = mean_squared_error(y_train_true_lr, y_train_pred_lr)
test_mse_lr = mean_squared_error(y_test_true_lr, y_test_pred_lr)
train_r2_lr = r2_score(y_train_true_lr, y_train_pred_lr)
test_r2_lr = r2_score(y_test_true_lr, y_test_pred_lr)

# In kết quả hồi quy tuyến tính
print("\nHồi quy tuyến tính:")
print(f"Tập huấn luyện - MSE: {train_mse_lr:.4f}, R²: {train_r2_lr:.4f}")
print(f"Tập kiểm tra - MSE: {test_mse_lr:.4f}, R²: {test_r2_lr:.4f}")


Hồi quy tuyến tính:
Tập huấn luyện - MSE: 0.1062, R²: 0.1768
Tập kiểm tra - MSE: 0.2274, R²: 0.2145


#### Nhận xét:
- **Hồi quy tuyến tính**:
  - Tập huấn luyện: MSE 0.1062, R² 0.1768 – hiệu suất thấp, giải thích ~18% biến thiên BEDAYNTM.
  - Tập kiểm tra: MSE 0.2274, R² 0.2145 – tốt hơn ANN trên test (R² 0.1846), nhưng vẫn kém.
- **So sánh với ANN** (train: MSE 0.0836, R² 0.3517; test: MSE 0.2361, R² 0.1846):
  - ANN tốt hơn trên train (R² cao hơn ~0.17), nhưng kém hơn trên test (R² thấp hơn ~0.03).
  - Hồi quy tuyến tính tổng quát hóa tốt hơn, ít overfitting hơn ANN.
- **Kết luận**: Cả hai mô hình đều có hiệu suất thấp, nhưng hồi quy tuyến tính đơn giản và ổn định hơn trên dữ liệu kiểm tra. Cần thêm đặc trưng hoặc điều chỉnh ANN để cải thiện.

# Bài tập 3.
Trong tệp dữ liệu `Real_estate.csv` chứa thông tin các giao dịch mua bán bất động sản.
Chúng ta có 414 mẫu dữ liệu, mỗi bản ghi có 8 cột theo thứ tự là

* **Cột x1**: Số thứ tự (chúng ta sẽ bỏ qua trường này)
* **Cột x2**: Ngày giao dịch mua bán (ta chỉ lấy phần nguyên là năm)
* **Cột x3**: Tuổi của căn nhà (theo năm)
* **Cột x4**: Khoảng cách tới ga MRT (phương tiện công cộng nội đô) gần nhất
* **Cột x5**: Số cửa hàng tiện ích gần đó
* **Cột x6**: Kinh độ căn nhà
* **Cột x7**: Vĩ độ căn nhà
* **Cột Y (đầu ra dự báo)**: Giá của căn nhà

Hãy chia dữ liệu thành phần training với 350 mẫu đầu tiên, phần validation với số mẫu còn lại. Hãy tham khảo các bài phần hồi quy tuyến tính và sử dụng mô hình ANN đã có để dự đoán Y đầu ra theo các cột từ X2 đến X6. Sau đó hãy chạy dự đoán cho phần dữ liệu validation và đưa ra tổng bình phương sai số của dự đoán. So sánh với phương pháp hồi quy tuyến tính.

In [23]:
# Đọc dữ liệu
data = pd.read_csv("Real_estate.csv")

# Xử lý cột ngày giao dịch: chỉ lấy năm
data["X1 transaction date"] = data["X1 transaction date"].apply(np.floor)

# Tách đặc trưng và nhãn
X = data[["X1 transaction date", "X2 house age", "X3 distance to the nearest MRT station", 
          "X4 number of convenience stores", "X5 latitude", "X6 longitude"]].values
y = data[["Y house price of unit area"]].values

# Chuẩn hóa dữ liệu
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Chia dữ liệu: 350 mẫu đầu cho train, còn lại cho validation
X_train = X_scaled[:350]
y_train = y_scaled[:350]
X_val = X_scaled[350:]
y_val = y_scaled[350:]

# Chuyển dữ liệu thành dạng ma trận
X_train = X_train.T  # Shape: (6, 350)
y_train = y_train.T  # Shape: (1, 350)
X_val = X_val.T      # Shape: (6, 64)
y_val = y_val.T      # Shape: (1, 64)

In [24]:
# Huấn luyện mô hình ANN
hidden_size = 10
num_iterations = 5000
learning_rate = 0.01
W1, b1, W2, b2 = train_model(X_train, y_train, hidden_size, num_iterations, learning_rate)

# Dự đoán ANN
y_val_pred_scaled = predict(X_val, W1, b1, W2, b2)
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.T).T
y_val_true = scaler_y.inverse_transform(y_val.T).T

# Tính SSE cho ANN trên validation
sse_ann = np.sum((y_val_true.T - y_val_pred.T) ** 2)
print("\nANN trên tập validation:")
print(f"Tổng bình phương sai số (SSE): {sse_ann:.4f}")

Iteration 0, Loss: 1.0244248431375305
Iteration 1000, Loss: 0.4336225594224574
Iteration 2000, Loss: 0.3807457264593743
Iteration 3000, Loss: 0.3703435134537972
Iteration 4000, Loss: 0.3528206281957679

ANN trên tập validation:
Tổng bình phương sai số (SSE): 3000.6539


In [25]:
# Hồi quy tuyến tính
model_lr = LinearRegression()
model_lr.fit(X_train.T, y_train.T)

# Dự đoán hồi quy tuyến tính
y_val_pred_scaled_lr = model_lr.predict(X_val.T)
y_val_pred_lr = scaler_y.inverse_transform(y_val_pred_scaled_lr)
y_val_true_lr = scaler_y.inverse_transform(y_val.T)

# Tính SSE cho hồi quy tuyến tính trên validation
sse_lr = np.sum((y_val_true_lr - y_val_pred_lr) ** 2)
print("\nHồi quy tuyến tính trên tập validation:")
print(f"Tổng bình phương sai số (SSE): {sse_lr:.4f}")


Hồi quy tuyến tính trên tập validation:
Tổng bình phương sai số (SSE): 4083.6448


#### Nhận xét:

- **ANN**:
  - Loss giảm đều từ 1.0244 xuống 0.3528, cho thấy mô hình học tốt trên tập huấn luyện.
  - SSE trên validation (3000.6539) khá thấp, chỉ ra dự đoán của ANN tương đối gần với giá trị thực tế.
- **Hồi quy tuyến tính**:
  - SSE trên validation (4083.6448) cao hơn ANN (~36% cao hơn), cho thấy dự đoán kém chính xác hơn.
- **So sánh**:
  - ANN vượt trội hơn hồi quy tuyến tính về độ chính xác trên tập validation, có thể do khả năng học các mối quan hệ phi tuyến giữa các đặc trưng và giá nhà.
  - Hồi quy tuyến tính đơn giản hơn nhưng không tận dụng được sự phức tạp của dữ liệu.
- **Kết luận**: ANN là lựa chọn tốt hơn cho bài toán này, nhưng cần kiểm tra thêm overfitting nếu R² hoặc MSE trên validation không cải thiện thêm.